# Deep Network Development, 2026 Autumn - Week 4 Practice - **Notebook for video recording**

## Sudoku - Base functionality


<img src='https://drive.google.com/uc?export=download&id=1S3J8J_-39VJLz_Tsq4lKxBFEyc1MFzP5' width='25%'>

Create a class called **SudokuBoard** that represents a Sudoku board. Sudoku is a puzzle game involving a 9×9 board divided into nine 3×3 blocks along the row and column boundaries. You need to place the numbers from 1 to 9 so that in every row, every column, and every 3×3 block each number appears exactly once. The internal representation of the board is up to you.

The class must provide the following member functions:

*   Create an initialization method that creates a `SudokuBoard` object representing an empty board.
*   `can_place(row_idx, col_idx, num) -> bool`: Returns a boolean indicating whether the number `num` can be placed at position (`row_idx`, `col_idx`) according to the rules of Sudoku. Row and column indexing starts from 0.
*   `place(pos, num) -> bool`: Returns a boolean indicating whether the number `num` can be placed at position `pos` according to the rules of Sudoku. `pos` is a tuple containing two integers between 0 and 8. Its first element is the row index, and its second element is the column index. If the function returns `True`, it must also place the given number on the board at the given position.


In [ ]:
import numpy as np

### Version 1

In [ ]:

class SudokuBoard:

    def __init__(self):
        self.board = np.zeros((9, 9), dtype=np.uint8)
        self.board_blockview = self.board.reshape(3, 3, 3, 3)   # (n_blockrow=3, n_row_in_block=3, n_blockcol=3, n_cols_in_block=3)

    def __str__(self):
        return str(self.board)

    def can_place(self, row_idx, col_idx, num):

        if self.board[row_idx, col_idx] != 0:
            return False

        if np.any(self.board[row_idx, :] == num):        # np.any(self.board[row_idx, :]) would be enough: np.any checks for True (non-False) values, and accordingly for non-zero values in numeric datatypes
            return False

        if np.any(self.board[:, col_idx] == num):
            return False

        block_start_row_idx = (row_idx // 3)*3
        block_start_col_idx = (col_idx // 3)*3

        if np.any(self.board[block_start_row_idx:block_start_row_idx+3,\
                   block_start_col_idx:block_start_col_idx+3] == num):
            return False

        return True

    def place(self, pos, num):

        row_idx, col_idx = pos
        success = self.can_place(row_idx, col_idx, num)
        if success is True:
            self.board[row_idx, col_idx] = num
        return success



### Version 2

Relying on an additional, block-based view of the board.

In [ ]:

class SudokuBoard:

    def __init__(self):
        self.board = np.zeros((9, 9), dtype=np.uint8)
        self.board_blockview = self.board.reshape(3, 3, 3, 3)   # (n_blockrow=3, n_row_in_block=3, n_blockcol=3, n_cols_in_block=3)

    def __str__(self):
        return str(self.board)

    def can_place(self, row_idx, col_idx, num):

        if self.board[row_idx, col_idx] != 0:
            return False

        if np.any(self.board[row_idx, :] == num):        # np.any(self.board[row_idx, :]) would be enough: np.any checks for True (non-False) values, and accordingly for non-zero values in numeric datatypes
            return False

        if np.any(self.board[:, col_idx] == num):
            return False

        block_row_idx, block_col_idx = row_idx // 3, col_idx // 3
        if np.any(self.board_blockview[block_row_idx, :, block_col_idx, :] == num):
            return False

        return True

    def place(self, pos, num):
        row_idx, col_idx = pos
        success = self.can_place(row_idx, col_idx, num)
        if success is True:
            self.board[row_idx, col_idx] = num
        return success


### Test cases

In [ ]:
import unittest

class TestSudoku(unittest.TestCase):
    def test_can_place(self):
        sb = SudokuBoard()
        self.assertTrue(sb.can_place(0, 0, 5))
        self.assertTrue(sb.can_place(8, 0, 5))
        self.assertTrue(sb.can_place(0, 8, 5))
        self.assertTrue(sb.can_place(4, 6, 5))
        self.assertTrue(sb.can_place(8, 8, 5))

        sb.place((3, 3), 5)
        self.assertFalse(sb.can_place(3, 3, 7))
        self.assertFalse(sb.can_place(5, 4, 5))
        self.assertFalse(sb.can_place(7, 3, 5))
        self.assertFalse(sb.can_place(3, 7, 5))

        self.assertTrue(sb.can_place(2, 4, 5))
        self.assertTrue(sb.can_place(3, 4, 6))

    def test_place(self):
        sb = SudokuBoard()
        self.assertTrue(sb.place((2, 7), 4))
        self.assertFalse(sb.place((2, 7), 4))
        self.assertFalse(sb.place((2, 7), 5))
        self.assertTrue(sb.place((2, 0), 5))
        self.assertFalse(sb.place((2, 8), 5))

    #

def suite():
    suite = unittest.TestSuite()
    testfuns = ["test_can_place", "test_place"]
    [suite.addTest(TestSudoku(fun)) for fun in testfuns]
    return suite

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite())


test_can_place (__main__.TestSudoku.test_can_place) ... ok
test_place (__main__.TestSudoku.test_place) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Sudoku - Extra methods

Additionally, implement the following member functions for the **SudokuBoard** class:
*   `num_numbers_in_blocks() -> ndarray(3, 3)`: Returns a 3 x 3 array of integers indicating the number of numbers placed into each 3 x 3 block of the board.
*   `select_rows_by_index(row_idxs) -> ndarray(k, 9)`: Returns a K x 9 array which is formed by the rows of the board indexed by `row_idxs`.
*   `select_cols_where_num_is_present(num) -> ndarray(9, k)`: Returns a 9 x k array which is formed by the columns of the board where the given number `num` is present.



In [ ]:

class SudokuBoard:

    def __init__(self):
        self.board = np.zeros((9, 9), dtype=np.uint8)
        self.board_blockview = self.board.reshape(3, 3, 3, 3)   # (n_blockrow=3, n_row_in_block=3, n_blockcol=3, n_cols_in_block=3)

    def __str__(self):
        return str(self.board)

    def can_place(self, row_idx, col_idx, num):

        if self.board[row_idx, col_idx] != 0:
            return False

        if np.any(self.board[row_idx, :] == num):
            return False

        if np.any(self.board[:, col_idx] == num):
            return False

        block_row_idx = (row_idx // 3)
        block_col_idx = (col_idx // 3)

        if np.any(self.board_blockview[block_row_idx, :, block_col_idx, :] == num):
            return False

        return True

    def place(self, pos, num):
        row_idx, col_idx = pos
        success = self.can_place(row_idx, col_idx, num)
        if success is True:
            self.board[row_idx, col_idx] = num
        return success

    # Extra methods

    def num_numbers_in_blocks(self):
        return np.count_nonzero(self.board_blockview, axis=(1, 3))   # (n_blockrow=3, n_row_in_block=3, n_blockcol=3, n_cols_in_block=3) -> (n_blockrow=3, n_blockcol=3)

    def select_rows_by_index(self, row_idxs):
        return self.board[row_idxs, :]

    def select_cols_where_num_is_present(self, num):
        is_num_present_in_cols_mask = np.any(self.board == num, axis=0)     # (n_rows=9, n_cols=9) bool_ -> (n_cols=9,)
        return self.board[:, is_num_present_in_cols_mask]



In [ ]:
my_board = SudokuBoard()

my_board.place(pos=(4, 7), num=9)
my_board.place(pos=(5, 7), num=8)
my_board.place(pos=(4, 8), num=2)

my_board.place(pos=(1, 2), num=9)

print(my_board)

print(my_board.num_numbers_in_blocks())

[[0 0 0 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 9 2]
 [0 0 0 0 0 0 0 8 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]]
[[1 0 0]
 [0 0 3]
 [0 0 0]]


In [ ]:
my_board = SudokuBoard()

my_board.place(pos=(4, 7), num=9)
my_board.place(pos=(5, 7), num=8)
my_board.place(pos=(4, 8), num=2)

my_board.place(pos=(1, 2), num=9)

print(my_board)
print("----")
print(my_board.select_rows_by_index([0,1,1,1,4,1,5]))

[[0 0 0 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 9 2]
 [0 0 0 0 0 0 0 8 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]]
----
[[0 0 0 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 9 2]
 [0 0 9 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 8 0]]


In [ ]:
my_board = SudokuBoard()

my_board.place(pos=(4, 7), num=9)
my_board.place(pos=(5, 7), num=8)
my_board.place(pos=(4, 8), num=2)

my_board.place(pos=(1, 2), num=9)

print(my_board)
print("----")
print(my_board.select_cols_where_num_is_present(9))
print("----")
print(my_board.select_cols_where_num_is_present(2))
print("----")
print(my_board.select_cols_where_num_is_present(7))
print(my_board.select_cols_where_num_is_present(7).shape)

[[0 0 0 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 9 2]
 [0 0 0 0 0 0 0 8 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]]
----
[[0 0]
 [9 0]
 [0 0]
 [0 0]
 [0 9]
 [0 8]
 [0 0]
 [0 0]
 [0 0]]
----
[[0]
 [0]
 [0]
 [0]
 [2]
 [0]
 [0]
 [0]
 [0]]
----
[]
(9, 0)
